In [ ]:
%%writefile /kaggle/working/my_agent.py
"""
LLM Visual Analyzer
====================
The LLM is the EYES. Specialists are the HANDS.

Phase 1: Heuristic exploration — record frame observations as JSONL.
Phase 2: Directed probing — wall mapping, click response analysis.
Phase 3: LLM classification — one inference call to identify game mechanics.
Phase 4: Route to best specialist for remaining budget.
Phase 5: On level-up or stuck — re-classify and adapt.

Backends: Kaggle transformers, local llama-cpp, heuristic fallback.
"""

import sys, os, json, time, random, argparse, traceback, threading
import numpy as np
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
os.environ.setdefault('HF_HUB_OFFLINE', '1')
from collections import deque, defaultdict
from pathlib import Path

# ---------------------------------------------------------------------------
# ACTION CONSTANTS
# ---------------------------------------------------------------------------
ACTION_RESET    = 0
ACTION_UP       = 1
ACTION_DOWN     = 2
ACTION_LEFT     = 3
ACTION_RIGHT    = 4
ACTION_INTERACT = 5
ACTION_CLICK    = 6
ACTION_UNDO     = 7

ACTION_NAMES  = {0:'RESET',1:'UP',2:'DOWN',3:'LEFT',4:'RIGHT',
                 5:'INTERACT',6:'CLICK',7:'UNDO'}
MOVE_DELTAS   = {1:(-1,0), 2:(1,0), 3:(0,-1), 4:(0,1)}
MOVE_ACTIONS  = list(MOVE_DELTAS.keys())

# ---------------------------------------------------------------------------
# SHARED CROSS-GAME KNOWLEDGE
# ---------------------------------------------------------------------------

_shared_knowledge = {
    'color_meanings':    {},   # color_int → 'player'|'target'|'wall'|'bg'
    'mechanic_priors':   {},   # game_prefix → game_type
    'solved_strategies': {},   # game_prefix → specialist_name
    'failed_approaches': {},   # game_prefix → list of failed specialists
}
_shared_lock = threading.Lock()


def update_shared(game_prefix, classification, solved=False, failed_specialist=None):
    with _shared_lock:
        gt = classification.get('game_type', 'unknown')
        sp = classification.get('suggested_specialist', 'random_explore')
        _shared_knowledge['mechanic_priors'][game_prefix] = gt
        if solved:
            _shared_knowledge['solved_strategies'][game_prefix] = sp
        if failed_specialist:
            lst = _shared_knowledge['failed_approaches'].setdefault(game_prefix, [])
            if failed_specialist not in lst:
                lst.append(failed_specialist)


def get_shared_prior(game_prefix):
    with _shared_lock:
        return _shared_knowledge['mechanic_priors'].get(game_prefix)


# ---------------------------------------------------------------------------
# HELPER FUNCTIONS
# ---------------------------------------------------------------------------

def to_2d(obs_or_array):
    if obs_or_array is None:
        return np.zeros((64, 64), dtype=np.int32)
    if hasattr(obs_or_array, 'frame'):
        f = np.array(obs_or_array.frame, dtype=np.int32)
    else:
        f = np.asarray(obs_or_array, dtype=np.int32)
    if f.ndim == 3: f = f[-1]
    if f.ndim != 2: f = np.zeros((64, 64), dtype=np.int32)
    return f


def bg_color(frame):
    return int(np.bincount(frame.flatten()).argmax())


def bfs_path(grid, start, goal, walls):
    if start == goal: return []
    h, w = grid.shape
    q, vis = deque([(start, [])]), {start}
    deltas = {1:(-1,0),2:(1,0),3:(0,-1),4:(0,1)}
    while q:
        pos, path = q.popleft()
        for act,(dr,dc) in deltas.items():
            n = (pos[0]+dr, pos[1]+dc)
            if (0<=n[0]<h and 0<=n[1]<w
                    and n not in vis and not walls[n[0],n[1]]):
                vis.add(n); new = path+[act]
                if n == goal: return new
                q.append((n, new))
    return []


def lawnmower_scan(w=64, h=64, step=8):
    pos = []
    for i, y in enumerate(range(0, h, step)):
        xs = range(0, w, step) if i%2==0 else range(w-1,-1,-step)
        for x in xs: pos.append((x, y))
    return pos


def gf2_solve(toggle_matrix, target_state):
    n, m = len(target_state), len(toggle_matrix)
    if m == 0 or n == 0: return []
    aug = np.zeros((n, m+1), dtype=int)
    for i in range(n):
        for j in range(m): aug[i,j] = int(toggle_matrix[j][i])%2
        aug[i,m] = int(target_state[i])%2
    pivot_cols, row = [], 0
    for col in range(m):
        found = next((r for r in range(row,n) if aug[r,col]==1), -1)
        if found==-1: continue
        aug[[row,found]] = aug[[found,row]]
        for r in range(n):
            if r!=row and aug[r,col]==1: aug[r]=(aug[r]+aug[row])%2
        pivot_cols.append(col); row+=1
    sol = np.zeros(m, dtype=int)
    for i,col in enumerate(pivot_cols):
        if i<n: sol[col] = aug[i,m]
    return [j for j in range(m) if sol[j]==1]


def detect_player_pos(prev_frame, cur_frame):
    diff = (prev_frame != cur_frame)
    if not diff.any(): return None
    bg = bg_color(prev_frame)
    rows, cols = np.where(diff)
    for r, c in zip(rows, cols):
        if prev_frame[r,c]==bg and cur_frame[r,c]!=bg:
            return (int(r), int(c))
    return (int(rows.mean()), int(cols.mean()))


# ---------------------------------------------------------------------------
# GAME OBSERVER — JSONL OBSERVATION COLLECTOR
# ---------------------------------------------------------------------------

class GameObserver:
    """Collects structured JSONL observations during exploration phases."""

    def __init__(self):
        self.jsonl       = []
        self.action_eff  = defaultdict(list)
        self.click_effs  = []
        self.toggle_map  = {}
        self.player_trail= []
        self.player_color= None
        self._bg         = None
        self._phase      = 'A'

    # ------------------------------------------------------------------
    def start_phase(self, frame, avail):
        self._bg = bg_color(frame)
        colors = sorted(int(c) for c in np.unique(frame))
        bg_pct = int(100 * np.sum(frame == self._bg) / frame.size)
        has_move  = any(a in MOVE_DELTAS for a in avail)
        has_click = ACTION_CLICK in avail
        self.jsonl.append(json.dumps({
            "type": "phase_start", "phase": self._phase,
            "bg": self._bg, "bg_pct": bg_pct, "colors": colors,
            "has_move": has_move, "has_click": has_click, "avail": avail,
        }))

    def next_phase(self):
        self._phase = 'B'

    # ------------------------------------------------------------------
    def record(self, prev_frame, action, data, cur_frame, step, obs2=None):
        changed    = not np.array_equal(prev_frame, cur_frame)
        diff_count = int(np.sum(prev_frame != cur_frame))
        bg         = self._bg or bg_color(prev_frame)

        self.action_eff[action].append({'changed': changed, 'diff': diff_count})

        if action in MOVE_DELTAS and changed:
            diff_mask  = (prev_frame != cur_frame)
            appeared   = {int(cur_frame[r,c]) for r,c in zip(*np.where(diff_mask))
                          if prev_frame[r,c]==bg and cur_frame[r,c]!=bg}
            disappeared= {int(prev_frame[r,c]) for r,c in zip(*np.where(diff_mask))
                          if cur_frame[r,c]==bg and prev_frame[r,c]!=bg}
            moving = appeared & disappeared or appeared
            if moving:
                mc = min(moving)
                if self.player_color is None:
                    self.player_color = mc
                    self.jsonl.append(json.dumps({
                        "type": "player_detected",
                        "color": mc, "step": step,
                        "action": ACTION_NAMES.get(action, str(action)),
                    }))
                pos = detect_player_pos(prev_frame, cur_frame)
                if pos:
                    self.player_trail.append(pos)
                    if len(self.player_trail) <= 5:
                        self.jsonl.append(json.dumps({
                            "type": "player_pos",
                            "row": pos[0], "col": pos[1], "step": step,
                            "action": ACTION_NAMES.get(action, str(action)),
                        }))

        if action in MOVE_DELTAS and not changed:
            if self.player_trail:
                p = self.player_trail[-1]
                dr, dc = MOVE_DELTAS[action]
                wr, wc = p[0]+dr, p[1]+dc
                if 0 <= wr < 64 and 0 <= wc < 64:
                    self.jsonl.append(json.dumps({
                        "type": "wall_found",
                        "row": wr, "col": wc, "step": step,
                        "direction": ACTION_NAMES.get(action, str(action)),
                    }))

        if action == ACTION_CLICK and data:
            x, y = int(data.get('x',32)), int(data.get('y',32))
            r, c = min(y, prev_frame.shape[0]-1), min(x, prev_frame.shape[1]-1)
            cb, ca = int(prev_frame[r,c]), int(cur_frame[r,c])
            key = (x, y)
            self.toggle_map.setdefault(key, []).append(ca)
            is_toggle = len(set(self.toggle_map[key])) >= 2
            self.click_effs.append({
                "x":x,"y":y,"before":cb,"after":ca,
                "changed":changed,"diff":diff_count,"toggle":is_toggle,"step":step,
            })

        if obs2 is not None:
            lv = getattr(obs2, 'levels_completed', 0) or 0
            if lv > 0:
                self.jsonl.append(json.dumps({
                    "type": "level_up", "level": lv, "step": step,
                    "action": ACTION_NAMES.get(action, str(action)),
                }))

    # ------------------------------------------------------------------
    def finalize(self, last_frame):
        bg = bg_color(last_frame)
        for act in sorted(self.action_eff):
            effs = self.action_eff[act]
            n_chg = sum(1 for e in effs if e['changed'])
            avg_d = int(np.mean([e['diff'] for e in effs])) if effs else 0
            self.jsonl.append(json.dumps({
                "type": "action_summary",
                "action": ACTION_NAMES.get(act, str(act)),
                "trials": len(effs), "changed": n_chg, "avg_diff_px": avg_d,
            }))

        if self.click_effs:
            n_ok = sum(1 for e in self.click_effs if e['changed'])
            any_tg = any(e['toggle'] for e in self.click_effs)
            self.jsonl.append(json.dumps({
                "type": "click_summary",
                "total": len(self.click_effs), "changed": n_ok,
                "toggle_behavior": any_tg,
                "sample": self.click_effs[:2],
            }))

        if self.player_trail:
            steps = [abs(self.player_trail[i][0]-self.player_trail[i-1][0])
                     +abs(self.player_trail[i][1]-self.player_trail[i-1][1])
                     for i in range(1, len(self.player_trail))]
            avg_step = int(np.median(steps)) if steps else 0
            self.jsonl.append(json.dumps({
                "type": "player_summary",
                "color": self.player_color,
                "trail_len": len(self.player_trail),
                "avg_step_px": avg_step,
                "last_pos": list(self.player_trail[-1]),
            }))

        tgt_cands = []
        pc = self.player_color
        for color in np.unique(last_frame):
            if color == bg or (pc is not None and color == pc): continue
            positions = np.argwhere(last_frame == color)
            if 1 <= len(positions) <= 80:
                tgt_cands.append({"color": int(color), "count": len(positions),
                                   "pos": positions[0].tolist()})
        if tgt_cands:
            self.jsonl.append(json.dumps({
                "type": "target_candidates",
                "candidates": tgt_cands[:6],
            }))

    # ------------------------------------------------------------------
    def get_jsonl_text(self, max_lines=50):
        return '\n'.join(self.jsonl[-max_lines:])

    def get_state(self, last_frame):
        """Return (player_pos, targets, walls)."""
        bg = bg_color(last_frame)
        pc = self.player_color
        player = self.player_trail[-1] if self.player_trail else None

        # Build color → count map for non-bg, non-player colors
        color_counts = {}
        for color in np.unique(last_frame):
            if color == bg or (pc is not None and color == pc):
                continue
            cnt = int(np.sum(last_frame == color))
            if 1 <= cnt <= 100:
                color_counts[color] = cnt

        # Use rarest eligible color as the goal (most goal-like)
        targets = []
        if color_counts:
            goal_color = min(color_counts, key=color_counts.get)
            targets = [tuple(p) for p in np.argwhere(last_frame == goal_color)]
            # Also include other rare colors (up to 3 total) as fallback targets
            for c, cnt in sorted(color_counts.items(), key=lambda x: x[1]):
                if c == goal_color:
                    continue
                if len(targets) >= 25:
                    break
                targets.extend([tuple(p) for p in np.argwhere(last_frame == c)])

        walls = np.zeros((64, 64), dtype=bool)
        # Fill walls from JSONL wall_found records
        for obs in self.jsonl:
            try:
                rec = json.loads(obs)
            except Exception:
                continue
            if rec.get('type') == 'wall_found':
                wr, wc = rec['row'], rec['col']
                if 0 <= wr < 64 and 0 <= wc < 64:
                    walls[wr, wc] = True
        return player, targets, walls


# ---------------------------------------------------------------------------
# LLM CLASSIFIER
# ---------------------------------------------------------------------------

_SYSTEM_PROMPT = """You are analyzing an interactive pixel-based game from gameplay observations.

These games share structural DNA with classic retro games (Atari 2600, NES, Genesis, SNES era) — the same domain that DQN was originally designed for. Like those games, expect:

- Grid or pixel-based visuals (64x64 frames, 16 colors max)
- Discrete actions: directional movement (up/down/left/right), interact, click at coordinates, undo
- Deterministic mechanics: same action in same state = same outcome
- Level-based progression (6-10 levels per game)
- Common mechanics from retro games:
  * Navigation through mazes with walls and exits
  * Collectible items that disappear on contact
  * Toggle puzzles (clicking cells flips neighbors)
  * Push puzzles (move objects onto target squares)
  * Enemies with patrol patterns to avoid
  * Switches/keys that unlock doors or new areas
  * Sequence collection (items must be gathered in order)
  * Interaction chains (use item A on object B to unlock C)

Analyze the observations below and classify what kind of game this is, what the player should do, and which strategy would work best.

Think step by step:
1. What control type? (movement only, click only, or both?)
2. What moves when the agent acts? (player? objects? nothing?)
3. What do clicks/interactions do? (toggle? collect? nothing?)
4. Are there patterns? (cycling animations? patrol paths? toggle matrices?)
5. Based on 1-4, classify and recommend a strategy."""

_CLASSIFY_PROMPT = """\
JSONL OBSERVATIONS:
{jsonl}

Respond with ONLY valid JSON (no markdown, no extra text):
{{
  "game_type": "<type>",
  "player": {{"detected": true/false, "color": <int or null>, "approx_pos": [<row>,<col>] or null}},
  "targets": [{{"color": <int>, "pos": [<row>,<col>]}}],
  "walls": {{"color": <int or null>}},
  "click_effect": "local_toggle" | "collect" | "global_change" | "none" | "unknown",
  "move_effect": "player_moves" | "no_effect" | "unknown",
  "interact_effect": "useful" | "no_effect" | "unknown",
  "suggested_specialist": "bfs_navigate" | "lawnmower_click" | "gf2_toggle" | "push_solver" | "patrol_avoid" | "interact_chain" | "timing" | "dqn" | "random_explore",
  "confidence": 0.0
}}"""

_SPECIALIST_MAP_BY_TYPE = {
    'mover':      'bfs_navigate',
    'clearer':    'bfs_navigate',
    'flipper':    'gf2_toggle',
    'clicker':    'lawnmower_click',
    'collector':  'lawnmower_click',
    'pusher':     'push_solver',
    'fighter':    'dqn',
    'router':     'dqn',
    'painter':    'lawnmower_click',
    'chaser':     'patrol_avoid',
    'chained':    'interact_chain',
    'cyclic':     'timing',
    'unknown':    'random_explore',
}


class LLMClassifier:
    _instance      = None
    _init_lock     = threading.Lock()
    _classify_lock = threading.Lock()   # llama-cpp Llama is NOT thread-safe

    @classmethod
    def get(cls, model_path=None):
        with cls._init_lock:
            if cls._instance is None:
                cls._instance = cls(model_path)
        return cls._instance

    def __init__(self, model_path=None):
        self.backend   = 'none'
        self.llm       = None
        self.model     = None
        self.tokenizer = None
        self._init(model_path)

    def _init(self, model_path=None):
        if os.environ.get('LLM_VISUAL_NO_LLM', '').strip('"\' ').lower() in ('1', 'true', 'yes'):
            print('[LLM] disabled by LLM_VISUAL_NO_LLM env var (heuristic only)')
            return

        kaggle = ["/kaggle/input/qwen2-5-coder/transformers/1.5b-instruct/1",
                  "/kaggle/input/qwen2-5-coder/transformers/1.5b-instruct",
                  "/kaggle/input/qwen2-5-coder-1-5b/transformers/default/1"]
        for kp in kaggle:
            if os.path.exists(kp):
                try:
                    from transformers import AutoTokenizer, AutoModelForCausalLM
                    self.tokenizer = AutoTokenizer.from_pretrained(kp, local_files_only=True)
                    self.model = AutoModelForCausalLM.from_pretrained(
                        kp, local_files_only=True, torch_dtype='auto', device_map='auto')
                    self.backend = 'transformers'
                    print(f'[LLM] transformers loaded from {kp}')
                    return
                except Exception as e:
                    print(f'[LLM] transformers failed: {e}')

        candidates = [
            model_path,
            str(Path(__file__).parent.parent / 'models' /
                'qwen2.5-coder-1.5b-instruct-q4_k_m.gguf'),
        ]
        for p in candidates:
            if p and os.path.exists(p):
                try:
                    from llama_cpp import Llama
                    self.llm     = Llama(model_path=p, n_ctx=2048, n_threads=4, verbose=False)
                    self.backend = 'llama_cpp'
                    print(f'[LLM] llama-cpp loaded from {p}')
                    return
                except Exception as e:
                    print(f'[LLM] llama-cpp failed: {e}')
        print('[LLM] No model — heuristic fallback only')

    def _generate(self, user_prompt, system_prompt=None):
        sys_content = system_prompt or _SYSTEM_PROMPT
        if self.backend == 'llama_cpp':
            try:
                out = self.llm.create_chat_completion(
                    messages=[{"role": "system", "content": sys_content},
                               {"role": "user",   "content": user_prompt}],
                    max_tokens=400, temperature=0.1,
                    stop=['\n\n', '```', 'Note:'])
                return out['choices'][0]['message']['content']
            except Exception:
                full = sys_content + '\n\n' + user_prompt
                out = self.llm(full, max_tokens=400, temperature=0.1,
                               stop=['\n\n', '```', 'Note:'])
                return out['choices'][0]['text']
        if self.backend == 'transformers':
            import torch
            msgs = [{"role": "system", "content": sys_content},
                    {"role": "user",   "content": user_prompt}]
            text = self.tokenizer.apply_chat_template(
                msgs, tokenize=False, add_generation_prompt=True)
            inp  = self.tokenizer(text, return_tensors='pt').to(self.model.device)
            with torch.no_grad():
                out = self.model.generate(
                    **inp, max_new_tokens=400, temperature=0.1,
                    do_sample=False, pad_token_id=self.tokenizer.eos_token_id)
            return self.tokenizer.decode(
                out[0][inp['input_ids'].shape[1]:], skip_special_tokens=True)
        return None

    @staticmethod
    def _extract_json(text):
        if not text: return None
        s = text.find('{')
        if s == -1: return None
        depth = 0
        for i in range(s, len(text)):
            if text[i]=='{': depth+=1
            elif text[i]=='}':
                depth-=1
                if depth==0:
                    try: return json.loads(text[s:i+1])
                    except: pass
                    break
        try: return json.loads(text)
        except: return None

    def classify(self, jsonl_text, avail):
        if self.backend != 'none':
            try:
                with LLMClassifier._classify_lock:
                    raw = self._generate(_CLASSIFY_PROMPT.format(jsonl=jsonl_text))
                result = self._extract_json(raw)
                if result:
                    gt = result.get('game_type','unknown')
                    sp = result.get('suggested_specialist')
                    if not sp:
                        result['suggested_specialist'] = _SPECIALIST_MAP_BY_TYPE.get(gt,'random_explore')
                    print(f'[LLM] classified: type={gt} '
                          f'specialist={result.get("suggested_specialist")} '
                          f'conf={result.get("confidence","?")}')
                    return result
            except Exception as e:
                print(f'[LLM] classify error: {e}')
        return self._heuristic(jsonl_text, avail)

    def _heuristic(self, jsonl_text, avail):
        has_move = has_click = has_toggle = False
        has_interact = ACTION_INTERACT in avail
        move_works = click_works = interact_works = reset_changes = False
        for line in jsonl_text.split('\n'):
            try:
                rec = json.loads(line)
            except Exception: continue
            t = rec.get('type','')
            if t in ('phase_start', 'frame_initial'):
                has_move  = rec.get('has_move', False)
                has_click = rec.get('has_click', False)
            if t == 'click_summary':
                has_toggle  = rec.get('toggle_behavior', False)
                click_works = rec.get('changed', 0) > 0
            if t == 'action_summary':
                nm = rec.get('action','')
                if nm in ('UP','DOWN','LEFT','RIGHT') and rec.get('changed',0) > 0:
                    move_works = True
                if nm == 'CLICK' and rec.get('changed',0) > 0:
                    click_works = True
                if nm == 'INTERACT' and rec.get('changed',0) > 0:
                    interact_works = True
                if nm == 'RESET' and rec.get('changed',0) > 0:
                    reset_changes = True

        if not has_move and not has_click:
            has_move  = any(a in MOVE_DELTAS for a in avail)
            has_click = ACTION_CLICK in avail

        if interact_works and has_interact:
            gt, sp = 'chained', 'interact_chain'
        elif reset_changes and not move_works and not click_works:
            gt, sp = 'cyclic', 'timing'
        elif has_click and has_toggle:
            gt, sp = 'flipper', 'gf2_toggle'
        elif has_move and not has_click:
            gt, sp = 'mover', 'bfs_navigate'
        elif has_click and not has_move:
            gt, sp = 'clicker', 'lawnmower_click'
        elif has_move and has_click:
            gt, sp = ('mover', 'bfs_navigate') if move_works else ('clicker', 'lawnmower_click')
        else:
            gt, sp = 'unknown', 'random_explore'

        return {'game_type':gt, 'suggested_specialist':sp,
                'confidence':0.45, '_source':'heuristic',
                'player':{'detected':False,'color':None,'approx_pos':None},
                'targets':[], 'walls':{'color':None}}


# ---------------------------------------------------------------------------
# SPECIALISTS
# ---------------------------------------------------------------------------

class NavigateSpecialist:
    def __init__(self, cls):
        self.cls        = cls
        self._stuck     = 0
        self._prev_pos  = None
        self._last_path = []
        self._dir_cycle = 0   # cycle directions when stuck

    def choose(self, frame, avail, turn, player, targets, walls):
        moves = [a for a in avail if a in MOVE_DELTAS]
        if not moves: return avail[0], None

        # No player detected: cycle directions systematically to find player
        if not player:
            act = moves[self._dir_cycle % len(moves)]
            self._dir_cycle += 1
            return act, None

        if not targets:
            act = moves[self._dir_cycle % len(moves)]
            self._dir_cycle += 1
            return act, None

        if self._prev_pos == player:
            self._stuck += 1
        else:
            self._stuck = 0
        self._prev_pos = player

        # When stuck: try a different direction to escape
        if self._stuck > 8:
            self._stuck = 0
            self._dir_cycle += 1
            return moves[self._dir_cycle % len(moves)], None

        near = min(targets, key=lambda t: abs(t[0]-player[0])+abs(t[1]-player[1]))
        dist = abs(near[0]-player[0])+abs(near[1]-player[1])
        if dist <= 1 and ACTION_INTERACT in avail: return ACTION_INTERACT, None
        if dist <= 2 and ACTION_CLICK in avail: return ACTION_CLICK, {'x':int(near[1]),'y':int(near[0])}

        path = bfs_path(frame, player, tuple(near), walls)
        if path:
            self._last_path = path
            return path[0], None

        # BFS blocked: move greedily toward target ignoring walls
        dr = near[0] - player[0]
        dc = near[1] - player[1]
        if abs(dr) >= abs(dc):
            preferred = 2 if dr > 0 else 1
        else:
            preferred = 4 if dc > 0 else 3
        if preferred in moves:
            return preferred, None
        return moves[self._dir_cycle % len(moves)], None


class LawnmowerSpecialist:
    def __init__(self, cls, step=8):
        self.positions = lawnmower_scan(64, 64, step)
        self._idx      = 0
        self._clicked  = set()

    def choose(self, frame, avail, turn, player, targets, walls):
        if ACTION_CLICK not in avail:
            moves = [a for a in avail if a in MOVE_DELTAS]
            return (random.choice(moves) if moves else avail[0]), None

        unclicked = [(int(t[1]),int(t[0])) for t in targets
                     if (int(t[1]),int(t[0])) not in self._clicked]
        if unclicked:
            x, y = unclicked[0]
            self._clicked.add((x, y))
            return ACTION_CLICK, {'x':x,'y':y}

        while self._idx < len(self.positions):
            x, y = self.positions[self._idx]
            self._idx += 1
            if (x,y) not in self._clicked:
                self._clicked.add((x,y))
                return ACTION_CLICK, {'x':x,'y':y}
        self._idx = 0
        x, y = self.positions[turn % len(self.positions)]
        return ACTION_CLICK, {'x':x,'y':y}


class GF2ToggleSpecialist:
    def __init__(self, cls):
        self._solution = None
        self._sol_idx  = 0
        self._fallback = LawnmowerSpecialist(cls, step=12)
        self._solved_once = False

    def _try_solve(self, frame):
        bg   = bg_color(frame)
        step = 8
        cells = [(r,c) for r in range(0,64,step) for c in range(0,64,step)]
        n    = len(cells)
        state  = [int(frame[r,c]!=bg) for r,c in cells]
        target = [0]*n
        idx_map = {(r,c):i for i,(r,c) in enumerate(cells)}
        toggle = []
        for r,c in cells:
            mask = [0]*n
            for dr,dc in [(0,0),(-step,0),(step,0),(0,-step),(0,step)]:
                nr,nc = r+dr, c+dc
                if (nr,nc) in idx_map: mask[idx_map[(nr,nc)]] = 1
            toggle.append(mask)
        idxs = gf2_solve(toggle, state)
        if idxs:
            return [(cells[i][1],cells[i][0]) for i in idxs]
        return None

    def choose(self, frame, avail, turn, player, targets, walls):
        if ACTION_CLICK not in avail:
            moves = [a for a in avail if a in MOVE_DELTAS]
            return (random.choice(moves) if moves else avail[0]), None

        if self._solution is None and (turn % 25 == 0):
            try:
                sol = self._try_solve(frame)
                if sol:
                    self._solution = sol
                    self._sol_idx  = 0
            except Exception: pass

        if self._solution and self._sol_idx < len(self._solution):
            x, y = self._solution[self._sol_idx]
            self._sol_idx += 1
            return ACTION_CLICK, {'x':int(x),'y':int(y)}
        return self._fallback.choose(frame, avail, turn, player, targets, walls)


class SokobanSpecialist:
    def __init__(self, cls):
        self.cls       = cls
        self._moves_q  = deque()
        self._fallback = NavigateSpecialist(cls)

    def choose(self, frame, avail, turn, player, targets, walls):
        moves = [a for a in avail if a in MOVE_DELTAS]
        if not moves: return avail[0], None

        # Execute queued push moves
        if self._moves_q:
            return self._moves_q.popleft(), None

        if not player: return random.choice(moves), None

        # Find pushable objects (non-bg clusters near player)
        bg = bg_color(frame)
        pc = None
        for r in range(max(0,player[0]-8), min(64,player[0]+8)):
            for c in range(max(0,player[1]-8), min(64,player[1]+8)):
                if frame[r,c]!=bg and (r,c)!=player:
                    pc = (r,c)
                    break
            if pc: break

        if pc and targets:
            near_goal = min(targets, key=lambda t: abs(t[0]-pc[0])+abs(t[1]-pc[1]))
            # Push box row by row toward goal
            if pc[0] > near_goal[0] and player[0] > pc[0]:
                self._moves_q.extend([1]*2)  # push box up: player must be below
            elif pc[0] < near_goal[0] and player[0] < pc[0]:
                self._moves_q.extend([2]*2)  # push box down
            elif pc[1] > near_goal[1] and player[1] > pc[1]:
                self._moves_q.extend([3]*2)  # push box left
            elif pc[1] < near_goal[1] and player[1] < pc[1]:
                self._moves_q.extend([4]*2)  # push box right
            if self._moves_q:
                return self._moves_q.popleft(), None

        return self._fallback.choose(frame, avail, turn, player, targets, walls)


class OnlineQSpecialist:
    """Simple online Q-table for unknown/complex game types."""

    def __init__(self, cls, n_states=128, lr=0.1, gamma=0.9, eps=0.3):
        self.n_states = n_states
        self.lr = lr; self.gamma = gamma; self.eps = eps
        self.Q = {}  # (state_id, act) → float
        self._prev_state = None
        self._prev_act   = None
        self._prev_level = 0

    def _encode(self, frame):
        bg = bg_color(frame)
        hist = np.bincount((frame // 1).flatten(), minlength=16).astype(float)
        hist /= max(1, hist.sum())
        h_idx = int(hist[1:].argmax())
        r_idx = int(np.sum(frame != bg)) % self.n_states
        return (h_idx * 10 + r_idx) % self.n_states

    def choose(self, frame, avail, turn, player, targets, walls):
        state = self._encode(frame)
        if random.random() < self.eps or state not in self.Q:
            act = random.choice(avail)
        else:
            q_vals = {a: self.Q.get((state,a), 0.0) for a in avail}
            act    = max(q_vals, key=q_vals.get)

        self._prev_state = state
        self._prev_act   = act

        if ACTION_CLICK in avail and act == ACTION_CLICK:
            if targets:
                t = targets[turn % len(targets)]
                return ACTION_CLICK, {'x':int(t[1]),'y':int(t[0])}
            positions = lawnmower_scan(64,64,step=12)
            x,y = positions[turn % len(positions)]
            return ACTION_CLICK, {'x':x,'y':y}
        return act, None

    def on_frame(self, frame, level):
        if self._prev_state is None: return
        reward = 10.0 if level > self._prev_level else -0.1
        self._prev_level = level
        state  = self._encode(frame)
        k      = (self._prev_state, self._prev_act)
        old    = self.Q.get(k, 0.0)
        best_next = max(self.Q.get((state,a),0.0) for a in range(8))
        self.Q[k] = old + self.lr*(reward + self.gamma*best_next - old)


class RandomExploreSpecialist:
    def __init__(self, cls=None): pass

    def choose(self, frame, avail, turn, player, targets, walls):
        if ACTION_CLICK in avail:
            if targets and turn % 2 == 0:
                t = targets[turn % len(targets)]
                return ACTION_CLICK, {'x':int(t[1]),'y':int(t[0])}
            positions = lawnmower_scan(64,64,step=12)
            x,y = positions[turn % len(positions)]
            return ACTION_CLICK, {'x':x,'y':y}
        moves = [a for a in avail if a in MOVE_DELTAS]
        return (random.choice(moves) if moves else avail[0]), None


class PatrolAvoidanceSpecialist:
    """Track moving objects, predict patrol paths, navigate between patrol windows."""

    def __init__(self, cls):
        self._npc_tracks = {}   # color → deque of centroids
        self._npc_vel    = {}   # color → (dr, dc) smoothed velocity
        self._wait       = 0
        self._nav        = NavigateSpecialist(cls)
        self._prev_frame = None

    def _npcs(self, frame, player):
        bg = bg_color(frame)
        out = {}
        for color in np.unique(frame):
            if color == bg: continue
            pos = np.argwhere(frame == color)
            if not (1 <= len(pos) <= 20): continue
            c = pos.mean(axis=0)
            if player and abs(c[0]-player[0]) < 2 and abs(c[1]-player[1]) < 2:
                continue
            out[int(color)] = (float(c[0]), float(c[1]))
        return out

    def _update(self, frame, player):
        if self._prev_frame is None:
            self._prev_frame = frame; return
        prev = self._npcs(self._prev_frame, player)
        curr = self._npcs(frame, player)
        for color, cp in curr.items():
            self._npc_tracks.setdefault(color, deque(maxlen=8)).append(cp)
            if color in prev:
                dr = cp[0] - prev[color][0]; dc = cp[1] - prev[color][1]
                if abs(dr) > 0.4 or abs(dc) > 0.4:
                    old = self._npc_vel.get(color, (dr, dc))
                    if old[0]*dr < 0 or old[1]*dc < 0:
                        self._npc_vel[color] = (dr, dc)
                    else:
                        self._npc_vel[color] = (0.7*old[0]+0.3*dr, 0.7*old[1]+0.3*dc)
        self._prev_frame = frame

    def _danger(self, pos):
        for color, vel in self._npc_vel.items():
            tr = self._npc_tracks.get(color)
            if not tr: continue
            p = (tr[-1][0]+vel[0], tr[-1][1]+vel[1])
            if abs(p[0]-pos[0]) < 2 and abs(p[1]-pos[1]) < 2: return True
        return False

    def choose(self, frame, avail, turn, player, targets, walls):
        moves = [a for a in avail if a in MOVE_DELTAS]
        if not moves: return avail[0], None
        self._update(frame, player)
        if player and self._danger(player): self._wait = 4
        if self._wait > 0:
            self._wait -= 1
            safe = [a for a in moves if player and
                    not self._danger((player[0]+MOVE_DELTAS[a][0], player[1]+MOVE_DELTAS[a][1]))]
            if safe: return safe[turn % len(safe)], None
        if player and targets:
            near = min(targets, key=lambda t: abs(t[0]-player[0])+abs(t[1]-player[1]))
            dw = walls.copy()
            for color, vel in self._npc_vel.items():
                tr = self._npc_tracks.get(color)
                if not tr: continue
                for step in range(1, 4):
                    pr, pc = int(tr[-1][0]+vel[0]*step), int(tr[-1][1]+vel[1]*step)
                    if 0 <= pr < 64 and 0 <= pc < 64: dw[pr, pc] = True
            path = bfs_path(frame, player, tuple(near), dw)
            if path: return path[0], None
        return self._nav.choose(frame, avail, turn, player, targets, walls)


class InteractionChainSpecialist:
    """Try ACTION5 near unique objects, track dependency chain, replay on game-over."""

    def __init__(self, cls):
        self._tried  = set()
        self._chain  = []
        self._cidx   = 0
        self._scan   = [(r, c) for r in range(4, 64, 8) for c in range(4, 64, 8)]
        self._sidx   = 0
        self._nav    = NavigateSpecialist(cls)

    def _rare_cells(self, frame):
        bg = bg_color(frame)
        cells = []
        for color in np.unique(frame):
            if color == bg: continue
            pos = np.argwhere(frame == color)
            if 1 <= len(pos) <= 10:
                cells.extend((int(p[0]), int(p[1])) for p in pos)
        return cells

    def _approach_and_act(self, frame, avail, player, walls, cell):
        if player:
            dist = abs(cell[0]-player[0]) + abs(cell[1]-player[1])
            if dist <= 1:
                if ACTION_INTERACT in avail: return ACTION_INTERACT, None
                if ACTION_CLICK in avail: return ACTION_CLICK, {'x':int(cell[1]),'y':int(cell[0])}
            path = bfs_path(frame, player, cell, walls)
            if path: return path[0], None
        if ACTION_CLICK in avail: return ACTION_CLICK, {'x':int(cell[1]),'y':int(cell[0])}
        return None, None

    def choose(self, frame, avail, turn, player, targets, walls):
        if self._chain and self._cidx < len(self._chain):
            cell = self._chain[self._cidx]
            act, data = self._approach_and_act(frame, avail, player, walls, cell)
            if act is not None:
                if player and act == ACTION_INTERACT: self._cidx += 1
                return act, data
            self._cidx += 1
        rare = [c for c in self._rare_cells(frame) if c not in self._tried]
        if rare:
            target = min(rare, key=lambda c: (abs(c[0]-player[0])+abs(c[1]-player[1])) if player else 0)
            act, data = self._approach_and_act(frame, avail, player, walls, target)
            if act is not None:
                self._tried.add(target)
                if act == ACTION_INTERACT and target not in self._chain:
                    self._chain.append(target)
                return act, data
        if ACTION_INTERACT in avail and self._sidx < len(self._scan):
            cell = self._scan[self._sidx]; self._sidx += 1
            act, data = self._approach_and_act(frame, avail, player, walls, cell)
            if act is not None: return act, data
        self._sidx = (self._sidx + 1) % len(self._scan)
        return self._nav.choose(frame, avail, turn, player, targets, walls)

    def on_game_over(self):
        self._sidx = 0; self._cidx = 0


class TimingSpecialist:
    """Detect periodic cycles in the world; act only during safe phase windows."""

    def __init__(self, cls):
        self._hashes    = deque(maxlen=40)
        self._period    = None
        self._safe_ph   = 0
        self._obs       = 0
        self._obs_lim   = 32
        self._nav       = NavigateSpecialist(cls)

    @staticmethod
    def _fhash(frame):
        return int(np.sum(frame.astype(np.int64)) % (2**31))

    def _detect(self):
        h = list(self._hashes); n = len(h)
        for p in range(2, n // 2 + 1):
            if all(h[i] == h[i+p] for i in range(n - p - 1, max(0, n - 2*p - 1), -1)
                   if i+p < n):
                return p
        return None

    def choose(self, frame, avail, turn, player, targets, walls):
        self._hashes.append(self._fhash(frame))
        moves = [a for a in avail if a in MOVE_DELTAS]
        if self._obs < self._obs_lim:
            self._obs += 1
            if self._obs >= 8 and self._period is None:
                self._period = self._detect()
                if self._period:
                    self._safe_ph = self._obs % self._period
            return (moves[turn % len(moves)] if moves else avail[0]), None
        if self._period:
            if len(self._hashes) >= self._period * 2:
                d = self._detect()
                if d and d != self._period:
                    self._period = d
            phase = turn % self._period
            if phase == self._safe_ph:
                return self._nav.choose(frame, avail, turn, player, targets, walls)
            if ACTION_INTERACT in avail and turn % 7 == 0:
                return ACTION_INTERACT, None
            return (moves[turn % len(moves)] if moves else avail[0]), None
        return self._nav.choose(frame, avail, turn, player, targets, walls)


# Priors derived from analysis of retro game mechanics
# (Atari 2600, NES, Genesis, SNES, Game Boy era games)
_SPECIALIST_CLASSES = {
    'bfs_navigate':   NavigateSpecialist,
    'lawnmower_click':LawnmowerSpecialist,
    'gf2_toggle':     GF2ToggleSpecialist,
    'push_solver':    SokobanSpecialist,
    'dqn':            OnlineQSpecialist,
    'random_explore': RandomExploreSpecialist,
    'patrol_avoid':   PatrolAvoidanceSpecialist,
    'interact_chain': InteractionChainSpecialist,
    'timing':         TimingSpecialist,
}


# ---------------------------------------------------------------------------
# ADAPTIVE TEMPO CONTROLLER
# ---------------------------------------------------------------------------

class AdaptiveTempoController:
    def __init__(self, alpha=1.5):
        self.alpha = alpha
        self.noop_rem = 0; self.acts_burst = 0
        self.burst_target = self._levy()
        self.total_noops = 0; self.total_acts = 0
        self.prog_noop = 0; self.prog_act = 0

    def _levy(self):
        return max(1, int((1-random.random())**(-1/max(self.alpha-1,0.1))))

    def should_act(self):
        if self.noop_rem>0:
            self.noop_rem-=1; self.total_noops+=1; return False
        self.acts_burst+=1
        if self.acts_burst>=self.burst_target:
            self.noop_rem=self._levy(); self.acts_burst=0
            self.burst_target=self._levy()
            self.total_noops+=1; self.noop_rem-=1; return False
        self.total_acts+=1; return True


# ---------------------------------------------------------------------------
# MAIN AGENT
# ---------------------------------------------------------------------------

class LLMVisualAgent:
    """
    LLM-as-visual-analyzer.

    Phases (by turn count within each game instance):
      0..EXPLORE_A-1        : Phase 1 — random heuristic exploration
      EXPLORE_A             : RESET, switch to Phase 2 directed probing
      EXPLORE_A..EXPLORE_B-1: Phase 2 — directed probing (wall mapping, click effects)
      EXPLORE_B             : LLM classification call
      EXPLORE_B..end        : Specialist play
    """

    EXPLORE_A  = 80    # heuristic exploration turns before first RESET
    EXPLORE_B  = 80    # directed probe turns before LLM call
    STUCK_LIMIT= 100   # turns without progress before LLM revision

    def __init__(self, game_id, avail_actions=None):
        self.game_id     = game_id
        self.game_prefix = game_id.split('-')[0]
        self.avail       = avail_actions or [1,2,3,4,6]
        self.observer    = GameObserver()
        self.llm         = LLMClassifier.get()
        self.specialist  = None
        self._cls        = None
        self.tempo       = AdaptiveTempoController(alpha=1.8)
        self._phase      = 'A'          # 'A', 'B', 'play'
        self._inner_turn = 0            # turn within current phase
        self._levels     = 0
        self._no_progress= 0
        self._prev_frame = None
        self._click_positions = lawnmower_scan(64,64,step=16)
        self._click_idx  = 0

        # Check cross-game prior
        prior = get_shared_prior(self.game_prefix)
        if prior:
            self._phase = 'play'
            sp = _SPECIALIST_MAP_BY_TYPE.get(prior, 'random_explore')
            cls_dict = {'game_type':prior,'suggested_specialist':sp,
                        'confidence':0.6,'_source':'prior'}
            self._cls = cls_dict
            self.specialist = _SPECIALIST_CLASSES.get(sp, RandomExploreSpecialist)(cls_dict)
            print(f'[{self.game_id}] Reusing prior: {prior} -> {sp}')

    # ------------------------------------------------------------------
    def _explore_a_action(self, frame, avail, turn):
        """Phase A: cycle directions, test clicks systematically."""
        if ACTION_CLICK in avail and turn % 8 == 0:
            if self._click_idx < len(self._click_positions):
                x, y = self._click_positions[self._click_idx]
                self._click_idx += 1
            else:
                x, y = random.randint(0,63), random.randint(0,63)
            return (ACTION_CLICK, {'x':x,'y':y}, 'probe-click')
        moves = [a for a in avail if a in MOVE_DELTAS]
        if moves:
            pattern = [1,2,3,4,1,4,2,3,1,1,2,2]
            return (moves[turn % len(moves)], None, 'probe-dir')
        if ACTION_INTERACT in avail and turn % 5 == 0:
            return (ACTION_INTERACT, None, 'probe-interact')
        return (random.choice(avail), None, 'probe-rand')

    def _explore_b_action(self, frame, avail, turn):
        """Phase B: targeted probing — wall detection, click response depth."""
        player = self.observer.player_trail[-1] if self.observer.player_trail else None
        if player and turn < 40:
            dirs = [a for a in avail if a in MOVE_DELTAS]
            if dirs:
                return (dirs[turn % len(dirs)], None, 'wall-probe')
        if ACTION_CLICK in avail and self.observer.click_effs:
            ex_idx = turn % len(self.observer.click_effs)
            ex = self.observer.click_effs[ex_idx]
            return (ACTION_CLICK, {'x':ex['x'],'y':ex['y']}, 'click-probe')
        return self._explore_a_action(frame, avail, turn)

    def _do_classify(self, frame, avail):
        """Run LLM classification, instantiate specialist."""
        self.observer.finalize(frame)
        jsonl_text = self.observer.get_jsonl_text(max_lines=20)  # fit in 2048 ctx
        self._cls  = self.llm.classify(jsonl_text, avail)
        sp         = self._cls.get('suggested_specialist', 'random_explore')
        SpecClass  = _SPECIALIST_CLASSES.get(sp, RandomExploreSpecialist)
        self.specialist = SpecClass(self._cls)
        print(f'[{self.game_id}] strategy={sp} '
              f'type={self._cls.get("game_type","?")} '
              f'conf={self._cls.get("confidence","?")}')
        update_shared(self.game_prefix, self._cls)

    # Cycle of specialists to try when stuck (in order)
    _STUCK_CYCLE = [
        'bfs_navigate', 'patrol_avoid', 'interact_chain',
        'lawnmower_click', 'timing', 'gf2_toggle', 'random_explore',
    ]

    def _revise(self, frame, avail):
        """On stuck: rotate through specialist types, use LLM if backend available."""
        prev_sp = self._cls.get('suggested_specialist', '') if self._cls else ''

        # If LLM is available, re-classify with updated observations
        if self.llm.backend != 'none':
            print(f'[{self.game_id}] STUCK -> re-classify')
            self.observer.finalize(frame)
            jsonl_text = self.observer.get_jsonl_text(max_lines=20)
            new_cls = self.llm.classify(jsonl_text, avail)
            new_sp  = new_cls.get('suggested_specialist', 'random_explore')
            # If LLM suggests same specialist, force rotation
            if new_sp == prev_sp:
                idx = self._STUCK_CYCLE.index(prev_sp) if prev_sp in self._STUCK_CYCLE else -1
                new_sp = self._STUCK_CYCLE[(idx + 1) % len(self._STUCK_CYCLE)]
                new_cls['suggested_specialist'] = new_sp
            update_shared(self.game_prefix, new_cls, failed_specialist=prev_sp)
            self._cls = new_cls
        else:
            # Heuristic mode: rotate through specialists
            idx = self._STUCK_CYCLE.index(prev_sp) if prev_sp in self._STUCK_CYCLE else -1
            new_sp = self._STUCK_CYCLE[(idx + 1) % len(self._STUCK_CYCLE)]
            if self._cls:
                self._cls['suggested_specialist'] = new_sp

        print(f'[{self.game_id}] STUCK -> {new_sp}')
        SpecClass = _SPECIALIST_CLASSES.get(new_sp, RandomExploreSpecialist)
        self.specialist = SpecClass(self._cls or {})
        self._no_progress = 0

    # ------------------------------------------------------------------
    def choose_action(self, frame_data, turn):
        frame = to_2d(frame_data)
        avail = (list(frame_data.available_actions)
                 if hasattr(frame_data, 'available_actions') else self.avail)

        # Phase A: heuristic exploration
        if self._phase == 'A':
            if self._inner_turn == 0:
                self.observer.start_phase(frame, avail)
            if self._inner_turn < self.EXPLORE_A:
                result = self._explore_a_action(frame, avail, self._inner_turn)
                self._inner_turn += 1
                self._prev_frame = frame
                return result
            # Phase A complete → transition to Phase B (no reset needed)
            self._phase      = 'B'
            self._inner_turn = 0
            self.observer.next_phase()

        # Phase B: directed probing
        if self._phase == 'B':
            if self._inner_turn == 0:
                self.observer.start_phase(frame, avail)
            if self._inner_turn < self.EXPLORE_B:
                result = self._explore_b_action(frame, avail, self._inner_turn)
                self._inner_turn += 1
                self._prev_frame = frame
                return result
            # End Phase B → classify
            self._do_classify(frame, avail)
            self._phase      = 'play'
            self._inner_turn = 0
            # Fall through to play phase

        # Play phase
        player, targets, walls = self.observer.get_state(frame)
        if self.specialist is None:
            self.specialist = RandomExploreSpecialist()

        # Stuck detection
        if self._prev_frame is not None:
            if np.array_equal(frame, self._prev_frame):
                self._no_progress += 1
            else:
                self._no_progress = 0
        if self._no_progress > self.STUCK_LIMIT:
            self._revise(frame, avail)
            player, targets, walls = self.observer.get_state(frame)

        try:
            act, data = self.specialist.choose(frame, avail, turn, player, targets, walls)
        except Exception as e:
            print(f'[{self.game_id}] specialist error: {e}')
            moves = [a for a in avail if a in MOVE_DELTAS]
            act, data = (random.choice(moves) if moves else avail[0]), None

        # Feed to OnlineQ
        if isinstance(self.specialist, OnlineQSpecialist):
            self.specialist.on_frame(frame, self._levels)

        reason = type(self.specialist).__name__.lower()[:8]
        self._prev_frame = frame
        self._inner_turn += 1
        return (act, data, reason)

    # ------------------------------------------------------------------
    def observe(self, prev_frame, act, data, cur_frame, obs2=None):
        try:
            self.observer.record(
                to_2d(prev_frame), act, data, to_2d(cur_frame),
                self._inner_turn, obs2)
        except Exception:
            pass

    def on_level_up(self, frame):
        self._levels += 1
        # 30-action probe on level-up to re-learn targets
        if self._phase == 'play' and self._levels > 0:
            try:
                avail = self.avail
                self.observer.finalize(to_2d(frame))
                jsonl = self.observer.get_jsonl_text(20)
                new_cls = self.llm.classify(jsonl, avail)
                sp = new_cls.get('suggested_specialist', 'random_explore')
                SpecClass = _SPECIALIST_CLASSES.get(sp, RandomExploreSpecialist)
                self.specialist = SpecClass(new_cls)
                self._cls = new_cls
                self._no_progress = 0
                print(f'[{self.game_id}] level-up re-classify -> {sp}')
            except Exception:
                pass
        update_shared(self.game_prefix, self._cls or {}, solved=True)

    def on_game_over(self, frame):
        self._phase      = 'A'
        self._inner_turn = 0
        self._levels     = 0
        self._no_progress= 0
        self.observer    = GameObserver()
        self.specialist  = None
        self._click_idx  = 0
        # Keep _cls for cross-game learning


# ---------------------------------------------------------------------------
# _ReplayWrapper
# ---------------------------------------------------------------------------

class _ReplayWrapper:
    def __init__(self, inner):
        self._inner = inner
        self._history = deque(maxlen=2000)
        self._best_seq   = []
        self._replaying  = False
        self._replay_idx = 0

    def choose_action(self, frame, turn):
        if self._replaying and self._replay_idx < len(self._best_seq):
            entry = self._best_seq[self._replay_idx]
            self._replay_idx += 1
            self._history.append(entry)
            if self._replay_idx >= len(self._best_seq): self._replaying = False
            return entry
        result = self._inner.choose_action(frame, turn)
        self._history.append((result[0],
                               result[1] if len(result)>1 else None,
                               result[2] if len(result)>2 else ''))
        return result

    def observe(self, prev_frame, act, data, frame):
        if hasattr(self._inner, 'observe'):
            self._inner.observe(prev_frame, act, data, frame)

    def on_level_up(self, frame):
        self._best_seq   = list(self._history)
        self._replaying  = False
        self._replay_idx = 0
        if hasattr(self._inner, 'on_level_up'):
            self._inner.on_level_up(frame)

    def on_game_over(self, frame):
        if self._best_seq:
            self._replaying  = True
            self._replay_idx = 0
        self._history.clear()
        if hasattr(self._inner, 'on_game_over'):
            self._inner.on_game_over(frame)


def create_agent(game_id, avail_actions=None):
    """Factory for local_test.py and test_sdk_games.py --agent interface."""
    avail = avail_actions if avail_actions else [1, 2, 3, 4, 6]
    return _ReplayWrapper(LLMVisualAgent(game_id, avail))


# ---------------------------------------------------------------------------
# STANDALONE RUNNER
# ---------------------------------------------------------------------------

def _p(msg): print(msg, flush=True)


def play_game(env, game_id, budget=600):
    from arc_agi import GameAction, GameState
    amap = {1:GameAction.ACTION1,2:GameAction.ACTION2,3:GameAction.ACTION3,
            4:GameAction.ACTION4,5:GameAction.ACTION5,6:GameAction.ACTION6,
            7:GameAction.ACTION7,0:GameAction.RESET}

    obs       = env.reset()
    agent     = LLMVisualAgent(game_id)
    max_lvl   = 0
    acts      = 0
    prev_frame= None

    for turn in range(budget):
        if obs is None: break
        if hasattr(obs,'state') and str(obs.state) in ('GameState.WIN','WIN'): break
        frame = to_2d(obs)

        if prev_frame is not None:
            agent.observe(prev_frame, prev_act, prev_data, frame)

        result = agent.choose_action(obs, turn)
        act, data = result[0], result[1] if len(result)>1 else None

        ga = amap.get(act, GameAction.ACTION1)
        try:
            prev_obs = obs
            obs2     = env.step(ga, data) if act==6 and data else env.step(ga)
            acts += 1
        except Exception as e:
            _p(f'  [{game_id}] STEP ERROR: {e}')
            break

        lv = getattr(obs2,'levels_completed',0) or 0
        if lv > max_lvl:
            max_lvl = lv
            agent.on_level_up(to_2d(obs2))
        if getattr(obs2,'full_reset',False):
            agent.on_game_over(to_2d(obs2))
            obs2 = env.reset()

        prev_frame = frame
        prev_act   = act
        prev_data  = data
        obs = obs2

    won = obs is not None and hasattr(obs,'state') and str(obs.state) in ('GameState.WIN','WIN')
    _p(f'  [{game_id}] L{max_lvl} {"WIN" if won else ""} | {acts} acts')
    return {'game_id':game_id,'levels':max_lvl,'won':won,'actions':acts}


def main():
    proj = os.path.dirname(os.path.dirname(os.path.abspath(__file__)))
    sys.path.insert(0, proj)

    parser = argparse.ArgumentParser(description='LLM Visual Analyzer test runner')
    parser.add_argument('--game',   default=None)
    parser.add_argument('--limit',  type=int, default=0)
    parser.add_argument('--budget', type=int, default=600)
    args = parser.parse_args()

    from arc_agi import Arcade, OperationMode
    envs = os.path.join(proj,'arc-interactive-main','arc-interactive-main','environment_files')
    if not os.path.isdir(envs):
        envs = os.path.join(proj,'arc-interactive','environment_files')
    arcade = Arcade(operation_mode=OperationMode.OFFLINE, arc_api_key='', environments_dir=envs)
    games  = arcade.get_environments()
    _p(f'[LLMVisual] {len(games)} games')

    if args.game:
        game_ids = [g.game_id for g in games if args.game in g.game_id]
        if not game_ids: _p(f'Game {args.game!r} not found'); return
    else:
        game_ids = [g.game_id for g in games]
        if args.limit > 0: game_ids = game_ids[:args.limit]

    results = []; t0 = time.time()
    for gid in game_ids:
        try:
            env = arcade.make(gid)
            results.append(play_game(env, gid, budget=args.budget))
        except Exception as e:
            _p(f'  [{gid}] ERROR: {e}'); traceback.print_exc()
            results.append({'game_id':gid,'levels':0,'error':str(e)})

    n = len(results); elapsed = time.time()-t0
    won = sum(1 for r in results if r.get('won'))
    lv  = sum(1 for r in results if r.get('levels',0)>0)
    _p(f'\n{"="*60}')
    _p(f'LLM Visual Analyzer  ({n} games, {elapsed:.0f}s)')
    _p(f'{"="*60}')
    _p(f'Won: {won}/{n}  ({100*won/max(n,1):.1f}%)')
    _p(f'L1+: {lv}/{n}   ({100*lv/max(n,1):.1f}%)')
    for r in sorted(results,key=lambda x:-x.get('levels',0)):
        if r.get('levels',0)>0 or r.get('won'):
            _p(f'  {r["game_id"]}: L{r["levels"]} {"WIN" if r.get("won") else ""}'
               f' | {r.get("actions",0)} acts')

# ─── MyAgent: Agent API bridge ────────────────────────────────────────────────
import threading as _bridge_threading
import numpy as _bridge_np
from arcengine import GameAction as _GameAction, GameState as _GameState
from agents.agent import Agent


class MyAgent(Agent):
    """Delegates to inner notebook agent via create_agent()."""

    def __init__(self, game_id, *args, **kwargs):
        super().__init__(game_id, *args, **kwargs)
        avail = [a if isinstance(a, int) else a.value
                 for a in (self.available_actions or [1, 2, 3, 4, 6])]
        self._agent = create_agent(game_id, avail)
        self._prev_frame = None
        self._prev_act = 1
        self._prev_data = None
        self._prev_levels = 0

    def choose_action(self, frames, latest_frame):
        result = [_GameAction.ACTION1]

        def _run():
            try:
                frame = _bridge_np.asarray(latest_frame.frame)
                if frame.ndim == 3:
                    frame = frame[-1]  # last frame in (T,H,W) sequence
                frame = frame.astype(_bridge_np.uint8)
                cur_levels = getattr(latest_frame, 'levels_completed', 0) or 0
                level_up = cur_levels > self._prev_levels
                game_over_reset = (self._prev_levels > 0 and cur_levels == 0)
                if self._prev_frame is not None and hasattr(self._agent, 'observe'):
                    try:
                        self._agent.observe(
                            self._prev_frame, self._prev_act,
                            self._prev_data, frame)
                    except Exception:
                        pass
                if game_over_reset and hasattr(self._agent, 'on_game_over'):
                    try:
                        self._agent.on_game_over(frame)
                    except Exception:
                        pass
                if level_up and hasattr(self._agent, 'on_level_up'):
                    try:
                        self._agent.on_level_up(frame)
                    except Exception:
                        pass
                out = self._agent.choose_action(frame, getattr(self, 'turn', 0))
                act = out[0]
                data = out[1] if len(out) > 1 else None
                if act == 6 and data:
                    ga = _GameAction.ACTION6
                    ga.set_data({'x': int(data['x']), 'y': int(data['y'])})
                    result[0] = ga
                elif act in (1, 2, 3, 4):
                    result[0] = _GameAction.from_id(act)
                self._prev_frame = frame
                self._prev_act = act
                self._prev_data = data
                self._prev_levels = cur_levels
            except Exception as _e:
                import traceback as _tb
                print(f'[MyAgent] error: {_e}', flush=True)
                _tb.print_exc()

        t = _bridge_threading.Thread(target=_run)
        t.start()
        t.join(25)
        return result[0]

    def is_done(self, frames=None, latest_frame=None):
        return False


In [ ]:
!pip install --no-index --find-links \
    /kaggle/input/competitions/arc-prize-2026-arc-agi-3/arc_agi_3_wheels \
    arc-agi python-dotenv

import os

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    !curl --fail --retry 999 --retry-all-errors --retry-delay 5 \
          --retry-max-time 600 http://gateway:8001/api/games

    !cp -r /kaggle/input/competitions/arc-prize-2026-arc-agi-3/ARC-AGI-3-Agents \
           /kaggle/working/ARC-AGI-3-Agents

    !cp /kaggle/working/my_agent.py \
        /kaggle/working/ARC-AGI-3-Agents/agents/templates/my_agent.py

    with open('/kaggle/working/ARC-AGI-3-Agents/agents/__init__.py', 'w') as f:
        f.write("from typing import Type, cast\n"
                "from dotenv import load_dotenv\n"
                "from .agent import Agent, Playback\n"
                "from .swarm import Swarm\n"
                "from .templates.my_agent import MyAgent\n\n"
                "load_dotenv()\n\n"
                'AVAILABLE_AGENTS: dict[str, Type[Agent]] = {\n'
                '  "myagent": MyAgent,\n'
                '}\n')

    with open('/kaggle/working/ARC-AGI-3-Agents/.env', 'w') as f:
        f.write("SCHEME=http\n"
                "HOST=gateway\n"
                "PORT=8001\n"
                "ARC_API_KEY=test-key-123\n"
                "ARC_BASE_URL=http://gateway:8001/\n"
                "OPERATION_MODE=online\n"
                "ENVIRONMENTS_DIR=\n"
                "RECORDINGS_DIR=/kaggle/working/server_recording\n")

    !cd /kaggle/working/ARC-AGI-3-Agents && \
        MPLBACKEND=agg \
        python main.py --agent myagent

import pandas as pd
if not os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    submission = pd.DataFrame(
        data=[['1_0', '1', True, 1]],
        columns=['row_id', 'game_id', 'end_of_game', 'score'])
    submission.to_parquet('/kaggle/working/submission.parquet', index=False)
